<a href="https://colab.research.google.com/github/Vengalagagan/NLP/blob/main/2403A52222(B_09)NLP_Lab_16_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HUGGING FACE TUTORIAL

In [34]:
!pip install transformers datasets

# import librarirs

In [35]:
from transformers import pipeline

# SENTIMENT ANALYSIS

In [36]:
classifier = pipeline("sentiment-analysis")
result=classifier("HUGGING FACE IS VERY EASY TO USE!")
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9988467693328857}]


DATASET IMPORTING

In [37]:
from datasets import load_dataset
dataset = load_dataset("imdb")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


# USING BERT MODEL

tokenizer & model

In [38]:
from transformers import AutoTokenizer,AutoModel
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model=AutoModel.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenization

In [39]:
text = "Transfomer are powerful models"
inputs = tokenizer(text,return_tensors='pt')
print(inputs)

{'input_ids': tensor([[  101,  9099, 14876,  5017,  2024,  3928,  4275,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


In [40]:
outputs= model(**inputs)
cls_embedding = outputs.last_hidden_state[:,0,:]

print(cls_embedding.shape)

torch.Size([1, 768])


# Text Classification Using Hugging Face With Sample Data

In [41]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

In [42]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [43]:
texts = [
    "This service is excellent",
    "I really enjoyed using this",
    "Absolutely fantastic experience",
    "I dislike the quality",
    "This was a terrible choice",
    "Completely unsatisfied with the result"
]

labels = [1, 1, 1, 0, 0, 0]


In [44]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    outputs = model(**inputs)

    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding.detach().numpy()[0]


In [45]:
X = np.array([get_embedding(t) for t in texts])
y = np.array(labels)

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [47]:
clf = GaussianNB()
clf.fit(X_train, y_train)

GaussianNB()

In [48]:
y_pred = clf.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)

Predictions: [1 1]
Actual: [0 0]


In [49]:
test_text = "This product is really good"

test_emb = get_embedding(test_text).reshape(1, -1)

prediction = clf.predict(test_emb)

print("Prediction:", "Positive" if prediction[0] == 1 else "Negative")


Prediction: Positive
